# 🛠 Feature Engineering

This notebook documents the feature engineering pipeline for the NCAA prediction model. It covers weighting recent games, normalizing stats by opponent and home-court advantage, and preparing combined datasets for advanced modeling.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

INPUT_PATH = '../input'
OUTPUT_PATH = '../output'

## 1. Weighting Recent Games

Linear weighting is applied to emphasize performance toward the end of the season. Teams that peak heading into the tournament are often better predictors of tournament success than their season-long average.

### Mathematical Logic
The weight for each game is calculated as:
$$Weight = 1 + \frac{DayNum}{MaxDayNum}$$
This means a game on the final day of the season (Day 132/133) has roughly twice the weight ($W \approx 2.0$) of a game on the first day of the season ($W \approx 1.0$).

In [2]:
def weight_recent_games(df, stat_columns):
    """
    Applies linear weighting to game stats to emphasize games later in the season.
    """
    output = df.copy()
    output['Weight'] = 1 + (output['DayNum'] / output.groupby(['League', 'Season'])['DayNum'].transform('max'))

    for col in stat_columns:
        output[col] = output[col] * output['Weight']

    return output

def agg_weight(df, stat_columns):
    """
    Aggregates game data to the season/team level using a weighted average.
    """
    season_agg = df.groupby(['League', 'Season', 'TeamId']).apply(
        lambda x: (x[stat_columns].sum() / x['Weight'].sum()),
        include_groups=False
    ).reset_index()

    return season_agg

## 2. Normalization

Raw stats are misleading if not adjusted for the quality of the opponent and the location of the game.

### Opponent Normalization
We adjust team stats by dividing them by the average stats their opponents allowed throughout the season. This effectively creates a "strength-adjusted" metric.
$$AdjustedStat = \frac{TeamStat}{OpponentSeasonAverageAllowed}$$

### Home-Court Normalization
We calculate the average "home-court effect" for each stat across the league and subtract/add it based on whether the team was at home or away, effectively simulating all games on a neutral court.

In [3]:
def normalize_by_opponent(df, stat_columns):
    """
    Adjusts team stats based on the defensive/offensive strength of their opponents.
    """
    data_agg = agg_weight(df, stat_columns)
    
    opp_stats = df.merge(
        data_agg.rename(columns={'TeamId':'TeamId_against'} | {x:f"opp_{x}" for x in stat_columns}),
        on=["League", "Season", "TeamId_against"],
        how='left'
    )

    for col in stat_columns:
        if "_against" in col:
            opp_stats[col] = opp_stats[col] / opp_stats[f'opp_{col}']
        else:
            opp_stats[col] = opp_stats[col] / opp_stats[f'opp_{col}_against']

    output = opp_stats.drop(columns=[f"opp_{x}" for x in stat_columns])

    return output

def normalize_by_home_court(df, stat_columns):
    """
    Adjusts stats to remove the variance caused by home-court advantage.
    """
    group_by = ["League", "Season", "TeamId"]
    home_data = df[df['home_away']==1]
    away_data = df[df['home_away']==-1]

    home_data_agg = agg_weight(home_data, stat_columns).rename({stat:stat + "_home" for stat in stat_columns}, axis=1)
    away_data_agg = agg_weight(away_data, stat_columns).rename({stat:stat + "_away" for stat in stat_columns}, axis=1)

    output = df.merge(home_data_agg, on=group_by).merge(away_data_agg, on=group_by)

    for stat in stat_columns:
        effect = (output[f'{stat}_home'] -  output[f'{stat}_away']) / 2  
        output.loc[output['home_away'] == 1, stat] -= effect
        output.loc[output['home_away'] == -1, stat] += effect
        
    output = output[["League", "Season", "TeamId", "DayNum", "Weight"] + stat_columns]
    
    return output

## 3. Advanced Efficiency Metrics

Standard points-per-game stats are often distorted by pace. We calculate Offensive and Defensive Efficiency (points per possession) to get a clearer picture of team quality.

- **Possessions**: Estimated using the Hollinger formula: $FGA + 0.44 \times FTA - OR + TO$
- **OEFF**: $Score / Possessions$
- **DEFF**: $ScoreAgainst / PossessionsAgainst$
- **eFG%**: $(FGM + 0.5 \times 3PM) / FGA$ (Adjusts for the extra value of 3-pointers)

In [4]:
def create_new_stats(df):
    """
    Calculates advanced efficiency and rate stats (OEFF, DEFF, eFG, etc.).
    """
    output = df.copy()

    # Shooting
    output["FGper"] = output["FGM"] / output["FGA"]
    output["FG3per"] = output["FGM3"] / output["FGA3"]
    output["FTper"] = output["FTM"] / output["FTA"]
    output["FGper_against"] = output["FGM_against"] / output["FGA_against"]

    # Efficiency
    output["Possessions"] = output["FGA"] + 0.44 * output["FTA"] - output["OR"] + output["TO"]
    output["Possessions_against"] = output["FGA_against"] + 0.44 * output["FTA_against"] - output["OR_against"] + output["TO_against"]

    output["OEFF"] = output["Score"] / output["Possessions"]
    output["DEFF"] = output["Score_against"] / output["Possessions_against"]
    output["NET_EFF"] = output["OEFF"] - output["DEFF"]
    
    output["eFG"] = (output["FGM"] + 0.5 * output["FGM3"]) / output["FGA"]
    output["TS"] = output["Score"] / (2 * (output["FGA" ] + 0.44 * output["FTA"]))
    
    return output

## 4. Team-Level Behavioral Features

Beyond efficiency, we calculate features that capture team "character":
- **Score Variance**: Are they consistent or boom/bust?
- **Close Game Win %**: How do they perform in high-pressure situations?
- **NET_EFF Last 10**: Are they trending upwards entering the tournament?

In [5]:
def calculate_team_level_features(df):
    """
    Calculates team-level variance and close game performance metrics from game-level data.
    """
    temp_df = df.copy()
    grouped = temp_df.groupby(["League", "Season", "TeamId"])
    
    # Game Score Variance
    variance = grouped.agg(Score_Variance=('Score', 'std')).reset_index()
    
    # Close Game Win Percentage (<= 5 point diff)
    temp_df['Score_Diff'] = np.abs(temp_df['Score'] - temp_df['Score_against'])
    close_games = temp_df[temp_df['Score_Diff'] <= 5]
    close_win_per = close_games.groupby(["League", "Season", "TeamId"])['Win'].mean().reset_index().rename(columns={'Win': 'Close_Game_Win_Per'})
    
    # Recency Trend (Last 10 games)
    temp_df = temp_df.sort_values(by=["League", "Season", "TeamId", "DayNum"], ascending=[True, True, True, False])
    last_10 = temp_df.groupby(["League", "Season", "TeamId"]).head(10)
    recency = last_10.groupby(["League", "Season", "TeamId"]).agg(NET_EFF_Last_10=('NET_EFF', 'mean')).reset_index()
    
    return variance.merge(close_win_per, on=["League", "Season", "TeamId"], how='left').merge(recency, on=["League", "Season", "TeamId"], how='left')